# Multi-Confirmation Trading Bot (FULL)

**Default lookback = DAYS**. Works with **Binance** and **Kraken** OHLCV **without API keys** (public endpoints via CCXT). **Forex** uses **OANDA** and requires an API key.

What you get:
- Broker choice in the Config cell: `binance` | `kraken` | `forex`.
- Lookback by **days** (default) or **bars**.
- EMA crossover + ADX + ATR filters + Higher Timeframe (HTF) confirmation.
- Backtester and console Paper Trader inside this notebook.

> If needed, install dependencies in the next cell: `pandas`, `numpy`, `ccxt`, `oandapyV20`, `python-dotenv`.

1) Install dependencies

In [9]:
# If you need to install (uncomment the next line and run once):
# !pip install -q pandas numpy ccxt oandapyV20 python-dotenv

2) Configuration

In [ ]:
import os
import logging

# Mode: backtest | paper | live
MODE = os.getenv("MODE", "backtest").lower()

# Broker: binance | kraken | forex
BROKER = os.getenv("BROKER", "binance").lower()

# Asset
# Binance/Kraken examples: BTC/USDT, ETH/USDT
# OANDA (forex) examples: EUR_USD, GBP_USD, USD_JPY
SYMBOL = os.getenv("SYMBOL", "BTC/USDT")

# Timeframes
# Options: "1m" "5m" "15m" "30m" "1h" "4h" "1d"
LTF = os.getenv("LTF", "5m")    # trading timeframe
HTF = os.getenv("HTF", "1h")   # higher timeframe for bias

# Lookback selection (DEFAULT = days)
DATA_PULL_MODE = os.getenv("DATA_PULL_MODE", "days").lower()  # 'days' | 'bars'
# Bars (used when DATA_PULL_MODE='bars')
LTF_BARS = int(os.getenv("LTF_BARS", "1500"))
HTF_BARS = int(os.getenv("HTF_BARS", "1500"))
# Days (used when DATA_PULL_MODE='days')
BACKTEST_DAYS = int(os.getenv("BACKTEST_DAYS", "30"))
PAPER_LTF_DAYS = int(os.getenv("PAPER_LTF_DAYS", "30"))
PAPER_HTF_DAYS = int(os.getenv("PAPER_HTF_DAYS", "60"))

# Strategy params
EMA_FAST = int(os.getenv("EMA_FAST", "9"))
EMA_SLOW = int(os.getenv("EMA_SLOW", "21"))
ADX_PERIOD = int(os.getenv("ADX_PERIOD", "14"))
ADX_THRESHOLD = float(os.getenv("ADX_THRESHOLD", "25"))
ATR_PERIOD = int(os.getenv("ATR_PERIOD", "14"))
ATR_SL_MULT = float(os.getenv("ATR_SL_MULT", "1.8"))  # default 1.5
ATR_TP_MULT = float(os.getenv("ATR_TP_MULT", "2.5"))  # default 2.5

# Risk & capital
RISK_PCT = float(os.getenv("RISK_PCT", "5.0"))      # percent equity per trade
INITIAL_CAPITAL = float(os.getenv("INITIAL_CAPITAL", "100"))

# Paper mode options
PAPER_POLL_INTERVAL = int(os.getenv("PAPER_POLL_INTERVAL", "60"))
PAPER_MAX_TRADES = int(os.getenv("PAPER_MAX_TRADES", "0"))  # 0 = unlimited

# API keys (optional for Binance/Kraken data; required for OANDA)
BINANCE_API_KEY = os.getenv("BINANCE_API_KEY", "")
BINANCE_API_SECRET = os.getenv("BINANCE_API_SECRET", "")
KRAKEN_API_KEY = os.getenv("KRAKEN_API_KEY", "")
KRAKEN_API_SECRET = os.getenv("KRAKEN_API_SECRET", "")
OANDA_API_KEY = os.getenv("OANDA_API_KEY", "")
OANDA_ACCOUNT_ID = os.getenv("OANDA_ACCOUNT_ID", "")
OANDA_ENVIRONMENT = os.getenv("OANDA_ENVIRONMENT", "practice")  # practice | live

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger("bot")

print(
    "Config:\n"
    f"  Mode          : {MODE}\n"
    f"  Broker        : {BROKER}  Symbol: {SYMBOL}\n"
    f"  LTF/HTF       : {LTF} / {HTF}\n"
    f"  Lookback mode : {DATA_PULL_MODE}  LTF_BARS={LTF_BARS}  HTF_BARS={HTF_BARS}\n"
    f"                  BACKTEST_DAYS={BACKTEST_DAYS}  PAPER_L/H={PAPER_LTF_DAYS}/{PAPER_HTF_DAYS}\n"
    f"  Capital       : ${INITIAL_CAPITAL:,.2f}  Risk/Trade: {RISK_PCT}%\n"
)

Config:
  Mode          : backtest
  Broker        : binance  Symbol: BTC/USDT
  LTF/HTF       : 5m / 1h
  Lookback mode : days  LTF_BARS=1500  HTF_BARS=1500
                  BACKTEST_DAYS=30  PAPER_L/H=30/60
  Capital       : $100.00  Risk/Trade: 5.0%



3) Data Fetchers

In [11]:
import time
from datetime import datetime
import numpy as np
import pandas as pd

_TF_MIN = {"1m":1,"3m":3,"5m":5,"15m":15,"30m":30,"1h":60,"2h":120,"4h":240,"6h":360,"12h":720,"1d":1440}

def _ccxt_client(name: str, key: str = '', secret: str = ''):
    try:
        import ccxt
    except Exception as e:
        raise RuntimeError("ccxt is required. Install with: pip install ccxt") from e
    opts = {
        'enableRateLimit': True,
        'options': {'defaultType': 'spot', 'adjustForTimeDifference': True},
        'timeout': 20000,
    }
    if key:
        opts.update({'apiKey': key or None, 'secret': secret or None})
    return getattr(ccxt, name)(opts)

def _fetch_ccxt_ohlcv_paginated(ex, symbol: str, timeframe: str, candles: int) -> pd.DataFrame:
    per_call = 1000
    out = []
    since = None
    remaining = int(candles)
    while remaining > 0:
        batch = min(per_call, remaining)
        ohlcv = ex.fetch_ohlcv(symbol, timeframe, since=since, limit=batch)
        if not ohlcv:
            break
        out.extend(ohlcv)
        remaining -= len(ohlcv)
        last_ts = ohlcv[-1][0]
        since = last_ts + 1
        rl = getattr(ex, 'rateLimit', None)
        if rl:
            time.sleep(rl/1000.0)
        if len(ohlcv) < batch:
            break
    df = pd.DataFrame(out, columns=["timestamp","open","high","low","close","volume"]).astype({
        "open":float,"high":float,"low":float,"close":float,"volume":float
    })
    if df.empty:
        return df
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    return df.set_index("timestamp").sort_index()

def fetch_binance(symbol: str, timeframe: str, *, bars: int=None, days: int=None) -> pd.DataFrame:
    ex = _ccxt_client('binance', BINANCE_API_KEY, BINANCE_API_SECRET)
    if bars:
        return _fetch_ccxt_ohlcv_paginated(ex, symbol, timeframe, bars)
    mins = _TF_MIN.get(timeframe, 60)
    approx = max(500, int((days or 30) * 1440 / mins))
    return _fetch_ccxt_ohlcv_paginated(ex, symbol, timeframe, approx)

def fetch_kraken(symbol: str, timeframe: str, *, bars: int=None, days: int=None) -> pd.DataFrame:
    ex = _ccxt_client('kraken', KRAKEN_API_KEY, KRAKEN_API_SECRET)
    if bars:
        return _fetch_ccxt_ohlcv_paginated(ex, symbol, timeframe, bars)
    mins = _TF_MIN.get(timeframe, 60)
    approx = max(500, int((days or 30) * 1440 / mins))
    return _fetch_ccxt_ohlcv_paginated(ex, symbol, timeframe, approx)

def fetch_forex(symbol: str, timeframe: str, *, bars: int=None, days: int=None) -> pd.DataFrame:
    try:
        import oandapyV20
        import oandapyV20.endpoints.instruments as instruments
    except Exception as e:
        raise RuntimeError("oandapyV20 is required for forex. Install with: pip install oandapyV20") from e
    tf_map = {"1m":"M1","5m":"M5","15m":"M15","30m":"M30","1h":"H1","4h":"H4","1d":"D"}
    if not OANDA_API_KEY:
        raise RuntimeError("OANDA_API_KEY is required for BROKER='forex'.")
    client = oandapyV20.API(access_token=OANDA_API_KEY, environment=OANDA_ENVIRONMENT)
    count = bars or max(500, (days or 30) * 48)
    req = instruments.InstrumentsCandles(
        instrument=symbol,
        params={"granularity": tf_map.get(timeframe, "H1"), "count": int(count)}
    )
    client.request(req)
    rows = []
    for c in req.response.get("candles", []):
        if not c.get("complete", False):
            continue
        mid = c.get("mid") or {}
        rows.append({
            "timestamp": pd.to_datetime(c["time"]),
            "open": float(mid.get("o", 0)),
            "high": float(mid.get("h", 0)),
            "low":  float(mid.get("l", 0)),
            "close":float(mid.get("c", 0)),
            "volume": int(c.get("volume", 0))
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.set_index("timestamp").sort_index()

def fetch_demo_data(symbol: str, timeframe: str, *, days: int=365) -> pd.DataFrame:
    mins = _TF_MIN.get(timeframe, 60)
    n = max(300, int(days * 1440 / mins))
    rng = pd.date_range(end=datetime.now(), periods=n, freq=f"{mins}min")
    np.random.seed(42)
    lr = np.random.normal(0.0001, 0.002, n)
    block = max(1, n // 6)
    for i in range(0, n, block):
        lr[i:i+block] += np.random.choice([-1,1]) * 0.0003
    prices = 100 * np.exp(np.cumsum(lr))
    high = prices * (1 + np.abs(np.random.normal(0, 0.003, n)))
    low  = prices * (1 - np.abs(np.random.normal(0, 0.003, n)))
    open_ = np.roll(prices, 1); open_[0] = prices[0]
    vol = np.random.randint(1000, 50000, n).astype(float)
    df = pd.DataFrame({"open":open_,"high":high,"low":low,"close":prices,"volume":vol}, index=rng)
    return df

def get_data(symbol: str, timeframe: str, *, bars: int=None, days: int=None) -> pd.DataFrame:
    broker = (BROKER or '').lower()
    source = broker.upper()
    try:
        if broker == 'binance':
            df = fetch_binance(symbol, timeframe, bars=bars, days=days)
        elif broker == 'kraken':
            df = fetch_kraken(symbol, timeframe, bars=bars, days=days)
        elif broker == 'forex':
            if not OANDA_API_KEY:
                log.warning("OANDA key missing — returning DEMO synthetic data.")
                df = fetch_demo_data(symbol, timeframe, days=days or 365)
                source = 'DEMO'
            else:
                df = fetch_forex(symbol, timeframe, bars=bars, days=days)
        else:
            df = fetch_demo_data(symbol, timeframe, days=days or 365)
            source = 'DEMO'
    except Exception as e:
        log.error(f"Data fetch failed on {broker}: {e} — using DEMO synthetic.")
        df = fetch_demo_data(symbol, timeframe, days=days or 365)
        source = 'DEMO'

    candles = len(df)
    if candles:
        span_days = (df.index[-1] - df.index[0]).days
        print(
            f"Data pull {symbol:<12} {timeframe:<4} src: {source:<6} "
            f"candles: {candles:>5} span: {span_days}d "
            f"({df.index[0].strftime('%Y-%m-%d')} -> {df.index[-1].strftime('%Y-%m-%d')})"
        )
    else:
        print("Data pull returned 0 candles.")
    return df

4) Indicators

In [12]:
import numpy as np
import pandas as pd

def ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()

def calc_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    hl = df['high'] - df['low']
    hc = (df['high'] - df['close'].shift()).abs()
    lc = (df['low'] - df['close'].shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()

def calc_adx(df: pd.DataFrame, period: int = 14) -> pd.Series:
    up = df['high'].diff()
    down = -df['low'].diff()
    pdm = np.where((up > down) & (up > 0), up, 0.0)
    mdm = np.where((down > up) & (down > 0), down, 0.0)
    tr = calc_atr(df, period)
    pdi = 100 * pd.Series(pdm, index=df.index).ewm(span=period, adjust=False).mean() / tr
    mdi = 100 * pd.Series(mdm, index=df.index).ewm(span=period, adjust=False).mean() / tr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return dx.ewm(span=period, adjust=False).mean()

def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['ema_fast'] = ema(df['close'], EMA_FAST)
    df['ema_slow'] = ema(df['close'], EMA_SLOW)
    df['atr'] = calc_atr(df, ATR_PERIOD)
    df['atr_ma'] = df['atr'].rolling(ATR_PERIOD * 2).mean()
    df['adx'] = calc_adx(df, ADX_PERIOD)
    return df.dropna()

5) Signal Generation

In [13]:
import numpy as np
import pandas as pd

def generate_signals(ltf_df: pd.DataFrame, htf_df: pd.DataFrame) -> pd.DataFrame:
    df = ltf_df.copy()
    htf_trend = (htf_df['ema_fast'] > htf_df['ema_slow']).astype(int).reindex(df.index, method='ffill')
    cross_up = (df['ema_fast'] > df['ema_slow']) & (df['ema_fast'].shift() <= df['ema_slow'].shift())
    cross_down = (df['ema_fast'] < df['ema_slow']) & (df['ema_fast'].shift() >= df['ema_slow'].shift())
    trend_ok = df['adx'] > ADX_THRESHOLD
    vol_ok = df['atr'] > df['atr_ma']

    df['signal'] = 0
    df.loc[cross_up & trend_ok & vol_ok & (htf_trend == 1), 'signal'] = 1
    df.loc[cross_down & trend_ok & vol_ok & (htf_trend == 0), 'signal'] = -1

    sign = df['signal']
    df['sl'] = np.where(sign == 1, df['close'] - ATR_SL_MULT * df['atr'],
                        np.where(sign == -1, df['close'] + ATR_SL_MULT * df['atr'], np.nan))
    df['tp'] = np.where(sign == 1, df['close'] + ATR_TP_MULT * df['atr'],
                        np.where(sign == -1, df['close'] - ATR_TP_MULT * df['atr'], np.nan))
    return df

6) Backtester 


In [14]:
import pandas as pd

class Backtester:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.equity = INITIAL_CAPITAL
        self.trades = []
        self.equity_curve = []

    def run(self):
        pos = None
        entry_price = sl = tp = None
        entry_time = None
        for ts, row in self.df.iterrows():
            self.equity_curve.append({'timestamp': ts, 'equity': self.equity})
            if pos is not None:
                hit_sl = (pos == 1 and row['low'] <= sl) or (pos == -1 and row['high'] >= sl)
                hit_tp = (pos == 1 and row['high'] >= tp) or (pos == -1 and row['low'] <= tp)
                if hit_sl or hit_tp:
                    exit_price = sl if hit_sl else tp
                    pct = (exit_price - entry_price) / entry_price * pos
                    rb = ATR_SL_MULT * row['atr'] / max(entry_price, 1e-9)
                    pnl = self.equity * RISK_PCT / 100 * pct / max(rb, 1e-9)
                    self.equity += pnl
                    self.trades.append({
                        'entry_time': entry_time,
                        'exit_time': ts,
                        'direction': 'LONG' if pos == 1 else 'SHORT',
                        'entry_price': entry_price,
                        'exit_price': exit_price,
                        'sl': sl, 'tp': tp, 'pnl': pnl,
                        'result': 'WIN' if hit_tp else 'LOSS'
                    })
                    pos = None
            if pos is None and row['signal'] != 0:
                pos = row['signal']
                entry_price = row['close']
                entry_time = ts
                sl = row['sl']
                tp = row['tp']
        return self._stats()

    def _stats(self):
        if not self.trades:
            return {'error': 'No trades — try lowering ADX_THRESHOLD or extending lookback'}
        df = pd.DataFrame(self.trades)
        wins = df[df['result'] == 'WIN']
        losses = df[df['result'] == 'LOSS']
        eq = pd.DataFrame(self.equity_curve).set_index('timestamp')
        dd = (eq['equity'] - eq['equity'].cummax()) / eq['equity'].cummax() * 100
        ret = df['pnl'] / INITIAL_CAPITAL
        sh = (ret.mean()/ret.std()*np.sqrt(252)) if ret.std() > 0 else 0
        return {
            'total_trades': int(len(df)),
            'win_rate': round(len(wins)/len(df)*100, 2) if len(df) else 0.0,
            'total_return': round((self.equity-INITIAL_CAPITAL)/INITIAL_CAPITAL*100, 2),
            'final_equity': round(self.equity, 2),
            'sharpe_ratio': round(sh, 2),
            'max_drawdown': round(float(dd.min()) if not dd.empty else 0.0, 2),
            'avg_win': round(float(wins['pnl'].mean()) if len(wins) else 0.0, 2),
            'avg_loss': round(float(losses['pnl'].mean()) if len(losses) else 0.0, 2),
            'profit_factor': round(float(wins['pnl'].sum()/abs(losses['pnl'].sum())) if len(losses) and abs(losses['pnl'].sum())>0 else float('inf'), 2),
            'trades_df': df,
            'equity_curve': eq,
        }

7) Paper Trader

In [15]:
import time

class PaperTrader:
    def __init__(self):
        self.equity = INITIAL_CAPITAL
        self.position = None  # 'LONG' | 'SHORT' | None
        self.entry_price = None
        self.entry_time = None
        self.sl = None
        self.tp = None
        self.trade_log = []
        self.iteration = 0

    def _unrealised_pct(self, price: float) -> float:
        if self.position == 'LONG':
            return (price - self.entry_price) / self.entry_price * 100
        if self.position == 'SHORT':
            return (self.entry_price - price) / self.entry_price * 100
        return 0.0

    def _record_exit(self, ts, row, exit_price, hit_tp: bool):
        sign = 1 if self.position == 'LONG' else -1
        pct = (exit_price - self.entry_price) / self.entry_price * sign
        rb = ATR_SL_MULT * row['atr'] / max(self.entry_price, 1e-9)
        pnl = self.equity * RISK_PCT / 100 * pct / max(rb, 1e-9)
        self.equity += pnl
        self.trade_log.append({
            'entry_time': self.entry_time,
            'exit_time': ts,
            'direction': self.position,
            'entry_price': self.entry_price,
            'exit_price': exit_price,
            'sl': self.sl, 'tp': self.tp,
            'pnl': pnl,
            'result': 'WIN' if hit_tp else 'LOSS'
        })
        self.position = None
        self.entry_price = self.entry_time = self.sl = self.tp = None

    def run(self):
        tf_s = {'1m':60,'5m':300,'15m':900,'30m':1800,'1h':3600,'4h':14400,'1d':86400}
        poll = PAPER_POLL_INTERVAL or tf_s.get(LTF, 3600)
        print(f"Paper trader started — {BROKER.upper()} {SYMBOL} {LTF}\n"
              f"Virtual capital: ${INITIAL_CAPITAL:,.2f}  Poll: {poll}s  Ctrl+C to stop\n")
        while True:
            try:
                self.iteration += 1
                if DATA_PULL_MODE == 'bars':
                    ltf_raw = get_data(SYMBOL, LTF, bars=LTF_BARS)
                    htf_raw = get_data(SYMBOL, HTF, bars=HTF_BARS)
                else:
                    ltf_raw = get_data(SYMBOL, LTF, days=PAPER_LTF_DAYS)
                    htf_raw = get_data(SYMBOL, HTF, days=PAPER_HTF_DAYS)
                ltf = compute_indicators(ltf_raw)
                htf = compute_indicators(htf_raw)
                row = ltf.iloc[-1]
                prev = ltf.iloc[-2]
                ts = ltf.index[-1]
                htf_row = htf.iloc[-1]
                htf_bull = htf_row['ema_fast'] > htf_row['ema_slow']
                cross_up = (row['ema_fast'] > row['ema_slow']) and (prev['ema_fast'] <= prev['ema_slow'])
                cross_down = (row['ema_fast'] < row['ema_slow']) and (prev['ema_fast'] >= prev['ema_slow'])
                adx_ok = row['adx'] > ADX_THRESHOLD
                atr_ok = row['atr'] > row['atr_ma']

                if self.position:
                    hit_sl = (self.position=='LONG' and row['low'] <= self.sl) or (self.position=='SHORT' and row['high'] >= self.sl)
                    hit_tp = (self.position=='LONG' and row['high'] >= self.tp) or (self.position=='SHORT' and row['low'] <= self.tp)
                    if hit_sl or hit_tp:
                        self._record_exit(ts, row, self.sl if hit_sl else self.tp, hit_tp)
                        self._print_status(ts, row, htf_bull, cross_up, cross_down)
                        if PAPER_MAX_TRADES and len(self.trade_log) >= PAPER_MAX_TRADES:
                            print(f"Reached max trades ({PAPER_MAX_TRADES}). Stopping.")
                            break
                        time.sleep(poll); continue
                    self._print_status(ts, row, htf_bull, cross_up, cross_down)
                    time.sleep(poll); continue

                long_ok = cross_up and adx_ok and atr_ok and htf_bull
                short_ok = cross_down and adx_ok and atr_ok and (not htf_bull)
                if long_ok or short_ok:
                    self.position = 'LONG' if long_ok else 'SHORT'
                    self.entry_price = row['close']
                    self.entry_time = ts
                    sgn = 1 if long_ok else -1
                    self.sl = row['close'] - sgn * ATR_SL_MULT * row['atr']
                    self.tp = row['close'] + sgn * ATR_TP_MULT * row['atr']
                self._print_status(ts, row, htf_bull, cross_up, cross_down)
            except KeyboardInterrupt:
                print("\nStopped by user.")
                break
            except Exception as e:
                log.error(f"Loop error: {e}")
            time.sleep(poll)

    def _print_status(self, ts, row, htf_bull, cross_up, cross_down):
        total_pnl = self.equity - INITIAL_CAPITAL
        win_count = sum(1 for t in self.trade_log if t['result']=='WIN')
        n_trades = len(self.trade_log)
        win_rate = (win_count/n_trades*100) if n_trades else 0.0
        unreal = self._unrealised_pct(row['close']) if self.position else 0.0
        print("-"*64)
        print(f"Candle #{self.iteration} @ {ts.strftime('%Y-%m-%d %H:%M:%S')}  Price {row['close']:.5f}")
        print(f"EMA{EMA_FAST} {row['ema_fast']:.5f}  EMA{EMA_SLOW} {row['ema_slow']:.5f}  ADX {row['adx']:.2f}  ATR {row['atr']:.5f}")
        print(f"HTF bias: {'BULL' if htf_bull else 'BEAR'}  Cross: {'UP' if cross_up else ('DOWN' if cross_down else 'NONE')}  ")
        if self.position:
            print(f"Open {self.position}  Entry {self.entry_price:.5f}  SL {self.sl:.5f}  TP {self.tp:.5f}  Unrl {unreal:+.2f}%")
        else:
            print("Open position: None")
        print(f"Equity ${self.equity:,.2f}  TotalPnL {total_pnl:+.2f}  Trades {n_trades}  WinRate {win_rate:.1f}%")

8) Runner (Backtest / Paper)

In [16]:
from math import sqrt

def run_backtest():
    print(f"Fetching {SYMBOL}  LTF:{LTF}  HTF:{HTF}")
    if DATA_PULL_MODE == 'bars':
        ltf_raw = get_data(SYMBOL, LTF, bars=LTF_BARS)
        htf_raw = get_data(SYMBOL, HTF, bars=HTF_BARS)
    else:
        ltf_raw = get_data(SYMBOL, LTF, days=BACKTEST_DAYS)
        htf_raw = get_data(SYMBOL, HTF, days=max(BACKTEST_DAYS*2, BACKTEST_DAYS+7))
    ltf = compute_indicators(ltf_raw)
    htf = compute_indicators(htf_raw)
    df = generate_signals(ltf, htf)
    bt = Backtester(df)
    stats = bt.run()
    if 'error' in stats:
        print('WARNING:', stats['error'])
        return
    print("\n" + "="*40)
    print("BACKTEST RESULTS")
    print("="*40)
    print(f"Symbol         : {SYMBOL}")
    print(f"Timeframe      : {LTF} -> {HTF}")
    print(f"Total trades   : {stats['total_trades']}")
    print(f"Win rate       : {stats['win_rate']}%")
    print(f"Total return   : {stats['total_return']}%")
    print(f"Final equity   : ${stats['final_equity']}")
    print(f"Sharpe ratio   : {stats['sharpe_ratio']}")
    print(f"Max drawdown   : {stats['max_drawdown']}%")
    print(f"Profit factor  : {stats['profit_factor']}")
    print(f"Avg win        : ${stats['avg_win']}")
    print(f"Avg loss       : ${stats['avg_loss']}")
    print("="*40 + "\n")
    #print(stats['trades_df'].head(20).to_string(index=False))
    #display(stats['trades_df'])
    print(stats['trades_df'].to_string(index=False))

if MODE == 'backtest':
    run_backtest()
elif MODE == 'paper':
    PaperTrader().run()
else:
    print("Live trading scaffold not implemented. Use MODE='paper' to validate first.")

Fetching BTC/USDT  LTF:5m  HTF:1h


02:07:27 ERROR    Data fetch failed on binance: binance {"code":-2008,"msg":"Invalid Api-Key ID."} — using DEMO synthetic.


Data pull BTC/USDT     5m   src: DEMO   candles:  8640 span: 29d (2026-02-12 -> 2026-03-14)


02:07:28 ERROR    Data fetch failed on binance: binance {"code":-2008,"msg":"Invalid Api-Key ID."} — using DEMO synthetic.


Data pull BTC/USDT     1h   src: DEMO   candles:  1440 span: 59d (2026-01-13 -> 2026-03-14)

BACKTEST RESULTS
Symbol         : BTC/USDT
Timeframe      : 5m -> 1h
Total trades   : 32
Win rate       : 28.12%
Total return   : -42.77%
Final equity   : $57.23
Sharpe ratio   : -4.46
Max drawdown   : -50.34%
Profit factor  : 0.56
Avg win        : $5.93
Avg loss       : $-4.18

                entry_time                  exit_time direction  entry_price  exit_price         sl         tp       pnl result
2026-02-12 12:22:27.987389 2026-02-12 13:12:27.987389      LONG    95.979871   94.998873  94.998873  97.342368 -4.785696   LOSS
2026-02-12 16:02:27.987389 2026-02-12 18:47:27.987389      LONG    95.091034   94.202451  94.202451  96.325178 -4.813773   LOSS
2026-02-12 19:42:27.987389 2026-02-12 22:12:27.987389      LONG    95.590806   94.580940  94.580940  96.993397 -4.491414   LOSS
2026-02-13 09:17:27.987389 2026-02-13 09:47:27.987389      LONG    93.861207   95.249269  92.861802  95.249269  5.1